# Predictions Test Analysis

Analysis of model predictions on the NIST23 test set (9-fold cross-validation), joining:
- `data/predicted/nps_inchikey_nist23_nist23_cv{0..8}.zip` — model outputs and cosine similarity scores (one zip per CV fold)
- `data/proc/nist23/spec_df.pkl` — spectrum metadata
- `data/proc/nist23/mol_df.pkl` — molecule metadata
- `data/classyfire/nist23_mol_classyfire.pkl` — ClassyFire chemical taxonomy labels

All 9 CV folds are concatenated into a single DataFrame before analysis.

In [1]:
import gc
import pickle
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

DATA_ROOT = Path("../data")

## 1. Load Data

In [2]:
# Load and combine predictions from all 9 CV folds
cv_dfs = []
for cv_idx in range(9):
    zip_path = DATA_ROOT / f"predicted/nps_inchikey_nist23_nist23_cv{cv_idx}.zip"
    with zipfile.ZipFile(zip_path) as zf:
        pkl_name = next(n for n in zf.namelist() if n.endswith("predictions_test.pkl"))
        with zf.open(pkl_name) as f:
            cv_df = pickle.load(f)
    cv_df["cv_fold"] = cv_idx
    cv_dfs.append(cv_df)
    print(f"  cv{cv_idx}: {len(cv_df):,} rows")

pred_df = pd.concat(cv_dfs, ignore_index=True)
print(f"\nCombined pred_df: {pred_df.shape}  ({pred_df['spec_id'].nunique():,} unique spec_ids)")

with open(DATA_ROOT / "proc/nist23/spec_df.pkl", "rb") as f:
    spec_df = pickle.load(f)

with open(DATA_ROOT / "proc/nist23/mol_df.pkl", "rb") as f:
    mol_df = pickle.load(f)
mol_df = mol_df.drop(columns=["mol"], errors="ignore")

with open(DATA_ROOT / "classyfire/nist23_mol_classyfire.pkl", "rb") as f:
    cf_df = pickle.load(f)

print(f"spec_df:  {spec_df.shape}")
print(f"mol_df:   {mol_df.shape}")
print(f"cf_df:    {cf_df.shape}")

del cv_dfs
gc.collect()


  cv0: 85,660 rows
  cv1: 83,574 rows
  cv2: 85,237 rows
  cv3: 85,873 rows
  cv4: 84,535 rows
  cv5: 84,931 rows
  cv6: 85,025 rows
  cv7: 85,048 rows
  cv8: 84,754 rows

Combined pred_df: (764637, 11)  (527,138 unique spec_ids)
spec_df:  (1899597, 22)
mol_df:   (52787, 13)
cf_df:    (52787, 10)


0

In [3]:
pred_df.head(3)

In [4]:
spec_df.head(3)

In [5]:
mol_df.head(3)

## 2. Merge DataFrames

In [6]:
# Merge predictions with spectrum metadata
spec_meta_cols = [
    "spec_id", "mol_id", "prec_type", "inst_type", "frag_mode",
    "ion_mode", "dset", "ace", "nce", "ce_type",
]
df = pred_df.merge(spec_df[spec_meta_cols], on=["spec_id", "mol_id"], how="left")

# Merge with molecule metadata
mol_meta_cols = ["mol_id", "smiles", "formula", "mw", "exact_mw", "num_atoms", "num_bonds", "inchikey_s"]
df = df.merge(mol_df[mol_meta_cols], on="mol_id", how="left")

# Merge with ClassyFire labels
cf_meta_cols = ["mol_id", "superklass", "klass", "subklass", "direct_parent"]
df = df.merge(cf_df[cf_meta_cols], on="mol_id", how="left")

# Remove MoNA spectra (adduct-matching artifact skews wrecall to ~0)
df = df[df["dset"] != "mona23"].copy()

print(f"Merged df (NIST23 only): {df.shape}")
df.head(3)
del pred_df, spec_df, mol_df, cf_df
gc.collect()


Merged df (NIST23 only): (698676, 30)


0

## 1. Top / Bottom Predictions

In [20]:
display_cols = ["spec_id", "mol_id", "smiles", "formula", "prec_type", "inst_type", "cos_sim_0.01", "cos_hun"]

print("=== Top 10 predictions ===")
display(df.nlargest(10, "cos_sim_0.01")[display_cols].reset_index(drop=True))

print("=== Bottom 10 predictions ===")
display(df.nsmallest(10, "cos_sim_0.01")[display_cols].reset_index(drop=True))

=== Top 10 predictions ===


=== Bottom 10 predictions ===
